# Tutorial 02 — NASA Gateway-class NRHO

Build the Gateway 9:2 Near-Rectilinear Halo Orbit via pseudo-arclength continuation of
the L2 halo family. Verify against NASA's published Gateway spec: period 6.56 d, perilune
~3,200 km (low pass over the Moon's south pole), apolune ~70,000 km, near-stable
(Floquet ~2 vs deep-libration ~1000+).

_~50 seconds to run._

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ariadne
from ariadne.data.constants import EARTH_MOON, R_MOON
from ariadne.orbits.families import lyapunov_orbit_at_jacobi
from ariadne.orbits.differential_correction import monodromy
from ariadne.dynamics.cr3bp import propagate

MU = EARTH_MOON.mu; L_STAR = EARTH_MOON.L_star; T_DAYS = EARTH_MOON.T_star / 86400.0

In [ ]:
# One-line Gateway NRHO construction
nrho = ariadne.gateway_nrho()
print(f'period = {nrho.period * T_DAYS:.3f} d  (Gateway spec: 6.56)')
print(f'periodicity residual = {nrho.residual:.1e}')

In [ ]:
# Sample + measure perilune/apolune
sol = propagate(nrho.s0, (0.0, nrho.period), MU, t_eval=np.linspace(0.0, nrho.period, 800))
d_moon = np.sqrt((sol.y[0] - (1 - MU)) ** 2 + sol.y[1] ** 2 + sol.y[2] ** 2) * L_STAR
print(f'perilune = {d_moon.min():.0f} km   (alt {d_moon.min() - R_MOON:.0f} km over pole)')
print(f'apolune  = {d_moon.max():.0f} km')

In [ ]:
# Near-stability: NRHO Floquet vs deep L1 Lyapunov for scale
floq_nrho = float(np.max(np.abs(np.linalg.eigvals(monodromy(MU, nrho)))))
floq_lyap = float(np.max(np.abs(np.linalg.eigvals(monodromy(MU, lyapunov_orbit_at_jacobi(MU, 'L1', 3.16))))))
print(f'NRHO max Floquet = {floq_nrho:.2f}')
print(f'L1 Lyap (C=3.16) Floquet = {floq_lyap:.0f}')
print(f'NRHO is {floq_lyap / floq_nrho:.0f}x more stable -- THIS is why Gateway flies an NRHO.')

In [ ]:
# Visualize
fig = plt.figure(figsize=(11, 4.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot(sol.y[0] * L_STAR, sol.y[1] * L_STAR, sol.y[2] * L_STAR, lw=1.0, color='crimson')
ax1.scatter([(1 - MU) * L_STAR], [0], [0], s=100, c='gray', label='Moon')
ax1.set_xlabel('x [km]'); ax1.set_ylabel('y [km]'); ax1.set_zlabel('z [km]')
ax1.set_title('Gateway NRHO (3D, synodic frame)')
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(sol.t * T_DAYS, d_moon, color='crimson', lw=1.2)
ax2.axhline(R_MOON, color='gray', ls='--', label=f'Moon surface ({R_MOON} km)')
ax2.set_xlabel('time [d]'); ax2.set_ylabel('distance to Moon [km]')
ax2.set_title('Moon distance per period'); ax2.grid(alpha=0.3); ax2.legend()
plt.tight_layout(); plt.show()